In [1]:
import os
import time
import copy

import numpy       as     np
import matplotlib.pyplot as plt
from   PIL         import Image

import torch
import torch.nn    as     nn
from   torch.utils.data.sampler import SubsetRandomSampler
import torch.optim as optim
from torch.optim import lr_scheduler

import torchvision
from   torchvision import datasets, transforms, models

In [2]:
# load pre-trained ShuffleNetV2 ~8MB model
# model = models.shufflenet_v2_x1_0(pretrained=True)
# model = models.resnet50(pretrained=True)
# model = models.squeezenet1_0(pretrained=True)
model = models.mobilenet_v2(pretrained=False)

# can be vocal or noise
num_classes = 11

# replace fully-connected layer to match our dataset
# added 512 and dropout
model.fc    = nn.Sequential(nn.Linear(1024, 1024),
                            nn.ReLU(inplace=True),
                            nn.Dropout(0.2),
                            nn.Linear(1024, num_classes))

# use GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

# transform input images to match network
trans = transforms.Compose([
    transforms.Resize(224),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
#     transforms.Lambda(lambda x: torch.cat([x,x,x], 0))
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [11]:
# prepare dataset
dataset_dir = "/raid/project/shared/vocalizations/vocalpy/vocal_class"
train_dir = os.path.join(dataset_dir, "train")

# -- create training dataset1p
train_dataset = datasets.ImageFolder(train_dir, transform=trans)
class_names   = train_dataset.classes
num_classes   = len(class_names)
train_size    = len(train_dataset)
train_indices = list(range(train_size))
np.random.shuffle(train_indices)

print('train dataset has {} images'.format(train_size))
print('train dataset has {} classes:'.format(num_classes))
print(class_names)

# -- create testing dataset
test_dir      = os.path.join(dataset_dir, "test")
test_dataset  = datasets.ImageFolder(test_dir, transform=trans)
test_size     = len(test_dataset)
test_indices  = list(range(test_size))
np.random.shuffle(test_indices)

class_names   = test_dataset.classes
num_classes   = len(class_names)
train_size    = len(test_dataset)
print('test dataset has {} images'.format(train_size))
print('test dataset has {} classes:'.format(num_classes))
print(class_names)

# -- create dataloaders
train_sampler = SubsetRandomSampler(train_indices)
test_sampler  = SubsetRandomSampler(test_indices)

batch_size  = 32
dataloaders = {
    'train': torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, num_workers=4, sampler=train_sampler),
     'test': torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, num_workers=4, sampler=test_sampler)
}

dataset_sizes = {'train': len(train_dataset), 'test': len(test_dataset)}
dataset_sizes

train dataset has 53036 images
train dataset has 11 classes:
['chevron', 'complex', 'down_fm', 'flat', 'mult_steps', 'rev_chevron', 'short', 'step_down', 'step_up', 'two_steps', 'up_fm']
test dataset has 2065 images
test dataset has 11 classes:
['chevron', 'complex', 'down_fm', 'flat', 'mult_steps', 'rev_chevron', 'short', 'step_down', 'step_up', 'two_steps', 'up_fm']


{'train': 53036, 'test': 2065}

In [12]:
# constant for classes
classes = ('noise', 'vocal')

# helper function to show an image
# (used in the `plot_classes_preds` function below)
def matplotlib_imshow(img, one_channel=False):
    if one_channel:
        img = img.mean(dim=0)
#     img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    if one_channel:
        plt.imshow(npimg, cmap="gray")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))

In [1]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)
exp_lr_scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

NameError: name 'nn' is not defined

In [14]:
from torch.utils.tensorboard import SummaryWriter

# default `log_dir` is "runs" - we'll be more specific here
writer = SummaryWriter('./tb_exp')

In [15]:
# # get some random training images
# dataiter = iter(dataloaders['train'])
# images, labels = dataiter.next()

# # create grid of images
# img_grid = torchvision.utils.make_grid(images)

# # show images
# matplotlib_imshow(img_grid, one_channel=True)

# # write to tensorboard
# writer.add_image('sample training images', img_grid, 0)

# for n_iter in range(10000000000000000):
#     writer.add_scalar('Loss/train', np.random.random(), n_iter)
#     writer.add_scalar('Loss/test', np.random.random(), n_iter)
#     writer.add_scalar('Accuracy/train', np.random.random(), n_iter)
#     writer.add_scalar('Accuracy/test', np.random.random(), n_iter)
# writer.close()

In [16]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=25):
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    aaa = 0
    for epoch in range(num_epochs):
        writer.add_scalar('epoch', epoch, epoch)
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'test']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
                # track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
            if phase == 'train':
                scheduler.step(1-best_acc)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            writer.add_scalar('epoch_loss', epoch_loss, epoch)
            writer.add_scalar('epoch_acc', epoch_acc, epoch)
            
            print('{} Loss: {:.4f} Acc: {:.4f}'.format(
                phase, epoch_loss, epoch_acc))

            # deep copy the model
            if phase == 'test' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))
    print('Best val Acc: {:4f}'.format(best_acc))

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model

In [17]:
def visualize_model(model, num_images=6):
    was_training = model.training
    model.eval()
    images_so_far = 0
    fig = plt.figure()

    with torch.no_grad():
        for i, (inputs, labels) in enumerate(dataloaders['test']):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for j in range(inputs.size()[0]):
                images_so_far += 1
                ax = plt.subplot(num_images//2, 2, images_so_far)
                ax.axis('off')
                ax.set_title('predicted: {}'.format(class_names[preds[j]]))
                imshow(inputs.cpu().data[j])

                if images_so_far == num_images:
                    model.train(mode=was_training)
                    return
        model.train(mode=was_training)

In [18]:
model_ft = train_model(model, criterion, optimizer, exp_lr_scheduler, num_epochs=100)

RuntimeError: CUDA out of memory. Tried to allocate 2.00 MiB (GPU 0; 23.62 GiB total capacity; 5.03 GiB already allocated; 12.88 MiB free; 32.50 MiB cached)

In [19]:
torch.save(model_ft, "./mobile_class_.pth.tar")

NameError: name 'model_ft' is not defined